## 1. Image Representation

A grayscale digital image is represented as a 2D array of pixel intensities. If the image has $M \times N$ pixels, then the total number of samples is

$$L = M \times N$$

Each pixel is stored as a value in the range $0$ to $255$, which makes the image suitable for lossless compression experiments.

In [2]:
from PIL import Image
import numpy as np

## 2. Reading the Image

The input image is converted to grayscale so that each pixel is represented by a single intensity value. This gives a 1D sequence of samples:

$$p = [p_1, p_2, \dots, p_n]$$

where $n$ is the number of pixels.

In [3]:
# 1. Read Image
image = Image.open("cat.png").convert("L")

pixels = np.array(image).flatten()

print("Image shape:", image.size)
print("Number of pixels:", len(pixels))

Image shape: (510, 758)
Number of pixels: 386580


## 3. Preparing Pixel Values

The pixel sequence is converted to integers so that the compression algorithm can process discrete symbols efficiently.

$$p_i \in \{0,1,2,\dots,255\}$$

This step ensures compatibility with the dictionary-based encoding scheme.

In [4]:
# 2. Convert pixel values to integers
pixels = [int(p) for p in pixels]

## 4. LZW Encoding Principle

Lempel-Ziv-Welch (LZW) compression builds a dictionary of repeated patterns in the pixel stream. A new symbol is encoded only when the current sequence is not already known.

If the current sequence is $w$ and the next symbol is $a$, then the algorithm tests whether

$$w+a$$

exists in the dictionary. If it does, it extends the sequence; otherwise, it outputs the code of $w$ and adds $w+a$ as a new entry.

In [5]:
# 3. LZW Encoding
def lzw_encode(data):

    # Initial dictionary:
    # All possible grayscale pixel values: 0-255
    dictionary = {}

    for i in range(256):
        dictionary[(i,)] = i

    next_code = 256

    w = ()
    encoded = []

    for pixel in data:

        # Add new pixel to current sequence
        wc = w + (pixel,)

        if wc in dictionary:

            # Sequence already exists
            w = wc

        else:

            # Store code of existing sequence
            encoded.append(dictionary[w])

            # Add new sequence to dictionary
            dictionary[wc] = next_code
            next_code += 1

            # Start new sequence
            w = (pixel,)

    # Add remaining sequence
    if w:
        encoded.append(dictionary[w])

    return encoded, dictionary

## 5. Compression Step

Once the dictionary is initialized, the encoder scans the full pixel sequence and produces a list of codes that represent repeated patterns.

The output is a compressed stream of codes:

$$C = [c_1, c_2, \dots, c_k]$$

where each $c_i$ is a dictionary index.

In [6]:
# 4. Perform LZW Compression
encoded, dictionary = lzw_encode(pixels)

## 6. Original Bit Requirement

For an 8-bit grayscale image, each pixel requires 8 bits of storage. Therefore, for $n$ pixels, the original size is

$$B_{orig} = 8n$$

This gives the baseline used for measuring compression gain.

In [7]:
# 5. Compression Statistics
# Original grayscale image:
# Each pixel = 8 bits

original_bits = len(pixels) * 8

## 7. Bit Requirement for Encoded Codes

The number of bits needed for each LZW code depends on the largest code value. If the maximum code is $c_{max}$, then the required bits are

$$b = \max(8, \lceil \log_2(c_{max}+1) \rceil)$$

This is the number of bits used to store each compressed code.

In [8]:
# Find number of bits needed for largest LZW code

max_code = max(encoded)

bits_per_code = max(
    8,
    max_code.bit_length()
)

## 8. Compressed Size

The total compressed size is obtained by multiplying the number of generated codes by the bits required per code:

$$B_{comp} = k \times b$$

where $k$ is the number of encoded symbols and $b$ is the bits per code.

In [9]:
# Compressed size
compressed_bits = len(encoded) * bits_per_code

## 9. Compression Ratio

The compression ratio compares the original size to the compressed size:

$$R = \frac{B_{orig}}{B_{comp}}$$

A larger ratio indicates better compression performance.

In [10]:
# Compression ratio
compression_ratio = original_bits / compressed_bits

## 10. Compression Percentage

The percentage of reduction in size is calculated as:

$$P = \left(1 - \frac{B_{comp}}{B_{orig}}\right) \times 100\%$$

This shows how much smaller the compressed output is compared with the original data.

In [11]:
# Compression percentage
compression_percentage = (
    (1 - compressed_bits / original_bits) * 100
)

## 11. Interpretation of Results

The final output reports the original size, compressed size, compression ratio, and percentage reduction. These values help interpret the effectiveness of the LZW method on the chosen image.

The key relationship is:

$$\text{Compression Gain} \approx \frac{B_{orig} - B_{comp}}{B_{orig}} \times 100\%$$

This provides a practical measure of how much storage is saved.

In [12]:
# 6. Display Results
print("\n========== LZW Compression Statistics ==========")

print("Original Size (bits):",
      original_bits)

print("Number of Original Pixels:",
      len(pixels))

print("Number of LZW Codes:",
      len(encoded))

print("Dictionary Size:",
      len(dictionary))

print("Bits per LZW Code:",
      bits_per_code)

print("Compressed Size (bits):",
      compressed_bits)

print("Compression Ratio:",
      round(compression_ratio, 2), ":1")

print("Compression Percentage:",
      round(compression_percentage, 2), "%")


========== LZW Compression Statistics ==========
Original Size (bits): 3092640
Number of Original Pixels: 386580
Number of LZW Codes: 145298
Dictionary Size: 145553
Bits per LZW Code: 18
Compressed Size (bits): 2615364
Compression Ratio: 1.18 :1
Compression Percentage: 15.43 %
